In [3]:
import numpy as np

if not hasattr(np, 'object'):
    np.object = object

import scanpy as sc
import pandas as pd
import os
import glob
import pickle

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

import seaborn as sns

/var/folders/k3/vg_c2k5x4mbcx6ktwfvjsddw0000gn/T/ipykernel_98214/149737704.py:3: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'):


In [4]:
DATA_FOLDER = "./tmp"
os.makedirs(DATA_FOLDER, exist_ok=True)

MM_TFS_FNAME = "pax6_wt_tfs.txt"

DATABASES_GLOB = "hg38__*.feather"

MOTIF_ANNOTATIONS_FNAME = "motifs-v9-nr.hgnc-m0.001-o0.0.tbl"

# WT output files
WT_ADJACENCIES_FNAME = os.path.join(
    DATA_FOLDER,
    "wt_clusters_6_9_adjacencies.tsv"
)

WT_MODULES_FNAME = os.path.join(
    DATA_FOLDER,
    "wt_clusters_6_9_modules.p"
)

WT_MOTIFS_FNAME = os.path.join(
    DATA_FOLDER,
    "wt_clusters_6_9_motifs.csv"
)

WT_REGULONS_FNAME = os.path.join(
    DATA_FOLDER,
    "wt_clusters_6_9_regulons.p"
)

# Mutant output files
MUT_ADJACENCIES_FNAME = os.path.join(
    DATA_FOLDER,
    "mutant_clusters_6_9_adjacencies.tsv"
)

MUT_MODULES_FNAME = os.path.join(
    DATA_FOLDER,
    "mutant_clusters_6_9_modules.p"
)

MUT_MOTIFS_FNAME = os.path.join(
    DATA_FOLDER,
    "mutant_clusters_6_9_motifs.csv"
)

MUT_REGULONS_FNAME = os.path.join(
    DATA_FOLDER,
    "mutant_clusters_6_9_regulons.p"
)

N_SAMPLES = 200

In [5]:
practicedata = sc.read_h5ad("Stage18_Data.h5ad")


/Users/marinmarcillac/miniconda3/envs/scenic_env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


In [6]:
wt_adata = practicedata[
    (practicedata.obs['orig.ident'] == 'pax6_WT') &
    (practicedata.obs['seurat_clusters'].isin([6, 9]))
].copy()

mutant_adata = practicedata[
    (practicedata.obs['orig.ident'] == 'pax6_Mutant') &
    (practicedata.obs['seurat_clusters'].isin([6, 9]))
].copy()

In [7]:
print("Pax6 WT clusters 6 + 9:", wt_adata.shape)

print(
    wt_adata.obs['seurat_clusters']
    .value_counts()
    .sort_index()
)

print("\nPax6 Mutant clusters 6 + 9:", mutant_adata.shape)

print(
    mutant_adata.obs['seurat_clusters']
    .value_counts()
    .sort_index()
)

Pax6 WT clusters 6 + 9: (362, 22806)
6    128
9    234
Name: seurat_clusters, dtype: int64

Pax6 Mutant clusters 6 + 9: (450, 22806)
6    330
9    120
Name: seurat_clusters, dtype: int64


In [8]:
if hasattr(wt_adata.X, "toarray"):
    ex_matrix_wt = pd.DataFrame(
        wt_adata.X.toarray(),
        index=wt_adata.obs_names,
        columns=wt_adata.var_names
    )
else:
    ex_matrix_wt = pd.DataFrame(
        wt_adata.X,
        index=wt_adata.obs_names,
        columns=wt_adata.var_names
    )

In [9]:
if hasattr(mutant_adata.X, "toarray"):
    ex_matrix_mutant = pd.DataFrame(
        mutant_adata.X.toarray(),
        index=mutant_adata.obs_names,
        columns=mutant_adata.var_names
    )
else:
    ex_matrix_mutant = pd.DataFrame(
        mutant_adata.X,
        index=mutant_adata.obs_names,
        columns=mutant_adata.var_names
    )

In [10]:
print("WT expression matrix:", ex_matrix_wt.shape)
print("Mutant expression matrix:", ex_matrix_mutant.shape)

WT expression matrix: (362, 22806)
Mutant expression matrix: (450, 22806)


In [11]:
tf_names = load_tf_names(MM_TFS_FNAME)

print("Number of TFs:", len(tf_names))
print(tf_names[:10])

Number of TFs: 22806
['LOC101732307', 'pi16', 'dok1', 'LOC101730410', 'mrps26', 'm1ap', 'loxl3', 'bbc3', 'htra2', 'LOC116408404']


In [12]:
db_fnames = glob.glob(DATABASES_GLOB)

def name(fname):
    return os.path.splitext(
        os.path.basename(fname)
    )[0]

dbs = [
    RankingDatabase(
        fname=fname,
        name=name(fname)
    )
    for fname in db_fnames
]

In [13]:
print(f"Loaded {len(dbs)} ranking databases:")

for db in dbs:
    print(db.name)

Loaded 1 ranking databases:
hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings


In [ ]:
wt_adjacencies = grnboost2(
    expression_data=ex_matrix_wt,
    tf_names=tf_names,
    client_or_address=client,
    seed=42,
    verbose=True
)

preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


/Users/marinmarcillac/miniconda3/envs/scenic_env/lib/python3.10/site-packages/distributed/worker.py:4798: UserWarning: Large object of size 1.27 MiB detected in task graph: 
  (["('from-delayed-881213499e320305a158537d66bedee8 ... e8', 22805)"],)
Consider scattering large objects ahead of time
with client.scatter to reduce scheduler burden and 
keep data on workers

    future = client.submit(func, big_data)    # bad

    big_future = client.scatter(big_data)     # good
    future = client.submit(func, big_future)  # good
  warnings.warn(


In [ ]:
print(wt_adjacencies.head())
print("Number of WT adjacency relationships:", len(wt_adjacencies))